<h1 style="color:#1f77b4; font-family:'Times New Roman';">
<b>SVD From Scratch</b>
</h1>
<div style="font-family:'Times New Roman';">
numpy gives you SVD in one line but i want to see where U, &Sigma; and V actually come from. The trick is to use eigen decomposition, since SVD and eigenvectors are closely related.
</div>

In [1]:
import numpy as np

## The idea

The matrix A<sup>T</sup>A is symmetric, and symmetric matrices have nice clean eigenvectors. It turns out:

- the eigenvectors of A<sup>T</sup>A are the right singular vectors **V**
- the eigenvalues of A<sup>T</sup>A are the singular values **squared**, so the singular values are just their square roots
- once we have V and the singular values, each column of **U** is just A v / sigma

so the plan is, eigen decompose A<sup>T</sup>A, take square roots for the singular values, then build U from there.

In [2]:
# step 1: get the singular values and V from A^T A
def get_sigma_and_V(A):
    AtA = A.T @ A
    eigvals, eigvecs = np.linalg.eigh(AtA)

    # eigh returns ascending, flip to biggest first
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    V = eigvecs[:, order]

    # tiny negative eigenvalues can show up from rounding, clip them
    sigma = np.sqrt(np.clip(eigvals, 0, None))
    return sigma, V

In [8]:
# step 2: build U, one column at a time, as A v / sigma
def build_U(A, sigma, V):
    U = np.zeros((A.shape[0], len(sigma)))
    for i in range(len(sigma)):
        if sigma[i] > 1e-10:
            U[:, i] = (A @ V[:, i]) / sigma[i]
    return U

In [3]:
def svd_from_scratch(A):
    sigma, V = get_sigma_and_V(A)
    U = build_U(A, sigma, V)
    return U, sigma, V.T   # return V transposed to match numpy

Now lets test it on a small matrix and check it actually rebuilds A.

In [4]:
A = np.array([[3, 1, 1],
              [-1, 3, 1],
              [2, 0, 1]], dtype=float)

U, S, Vt = svd_from_scratch(A)
print('singular values:', S.round(3))

singular values: [3.954 3.341 0.454]


In [5]:
A_rebuilt = U @ np.diag(S) @ Vt
print('rebuilt A matches?', np.allclose(A, A_rebuilt))
A_rebuilt.round(2)

rebuilt A matches? True


array([[ 3.,  1.,  1.],
       [-1.,  3.,  1.],
       [ 2.,  0.,  1.]])

<h2 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Comparing with numpy</b>
</h2>
<div style="font-family:'Times New Roman';">
Lets line our singular values up against numpy's. The U and V vectors can come out with flipped signs (a direction and its opposite are the same axis), so those might not match exactly, but the singular values should.
</div>

In [6]:
U2, S2, Vt2 = np.linalg.svd(A)

print('ours :', S.round(4))
print('numpy:', S2.round(4))
print('singular values match?', np.allclose(S, S2))

ours : [3.9539 3.3407 0.4542]
numpy: [3.9539 3.3407 0.4542]
singular values match? True


### quick recap

- got V and the singular values from the eigen decomposition of A<sup>T</sup>A
- built U column by column as A v / sigma
- rebuilt the original matrix and matched numpy on the singular values

ofcourse numpy's version is faster and more stable for big matrices, but atleast now its not a mystery. Next i'll point this at an image and actually compress it.